In [ ]:
!pip install -q transformers datasets accelerate evaluate

In [ ]:
from datasets import load_dataset
from collections import Counter

In [ ]:
dataset = load_dataset("ccdv/arxiv-classification")
dataset

In [ ]:
def filter_cs_categories(dataset):
  allowed_labels = {
      1,  # cs.CV
      2,  # cs.AI
      3,  # cs.SY
      5,  # cs.CE
      6,  # cs.PL
      7,  # cs.IT
      8,  # cs.DS
      9   # cs.NE
  }

  for split in ["train", "validation", "test"]:
    dataset[split] = dataset[split].filter(lambda x: x["label"] in allowed_labels)

  return dataset

In [ ]:
dataset = filter_cs_categories(dataset)
dataset

In [ ]:
def remap_labels(dataset):
  remap_labels = {
      1: 0,
      2: 1,
      3: 2,
      5: 3,
      6: 4,
      7: 5,
      8: 6,
      9: 7
  }

  for split in ["train", "validation", "test"]:
    dataset[split] = dataset[split].map(lambda x: {"label": remap_labels[x["label"]]})

  return dataset

In [ ]:
dataset = remap_labels(dataset)
dataset

In [ ]:
from collections import Counter

Counter(dataset["train"]["label"])

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("answerdotai/ModernBERT-base")

In [ ]:
def tokenize_function(example):
    return tokenizer(example["text"], truncation=True, max_length=256)

In [ ]:
tokenized_dataset = dataset.map(tokenize_function, batched=True)

In [ ]:
print(tokenized_dataset["train"][0].keys())
print(len(tokenized_dataset["train"][0]["input_ids"]))

In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained("answerdotai/ModernBERT-base", num_labels=8)

model

In [ ]:
print(model.config.num_labels)

In [ ]:
print(model.classifier)

In [ ]:
tokenized_dataset = tokenized_dataset.remove_columns(["text"])

In [ ]:
print(tokenized_dataset["train"][0].keys())

In [ ]:
from transformers import TrainingArguments
training_args = TrainingArguments(
    output_dir="./modernbert_classifier",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    logging_steps=100,
    report_to="none"
)

In [ ]:
import evaluate
accuracy = evaluate.load("accuracy")

In [ ]:
import numpy as np

def compute_metrics(eval_pred):

    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=1)

    return accuracy.compute(
        predictions=predictions,
        references=labels
    )

In [ ]:
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

In [ ]:
trainer.train()

In [ ]:
results = trainer.evaluate()
print(results)

In [ ]:
test_results = trainer.evaluate(tokenized_dataset["test"])
print(test_results)

In [ ]:
trainer.save_model("modernbert_classifier")

In [ ]:
tokenizer.save_pretrained("modernbert_classifier")

In [ ]:
!zip -r modernbert_classifier.zip modernbert_classifier